# NS11 Tutorial A - Power Laws, Heavy Tails, and the Long Tail

**Lecture:** NS11 - Power Laws and Rich-Get-Richer Phenomena  
**Reading:** Clauset, Shalizi, and Newman (2009); Anderson (2006)

## Learning goals
- Distinguish a **broad / heavy-tailed** distribution from a specific **power-law** claim.
- Use **CCDF** and **rank-size** plots to study popularity data.
- Compare a real popularity distribution to narrow and broad baselines.
- Quantify concentration and explain what a **long tail** means in practice.

## Outline
1. Real popularity example: the OpenFlights world network.
2. Compare the real tail to Erdos-Renyi and BA baselines.
3. Why a straight log-log region is not enough to prove a power law.
4. Long tail as a head-vs-tail question.
5. Exercise and takeaway.


In [ ]:
from netsci_utils import *
import pandas as pd

set_seeds()


def sample_power_law(gamma=2.6, xmin=1.0, size=5000, seed=RANDOM_SEED):
    """Generate a continuous sample with PDF proportional to x^{-gamma}."""
    rng = np.random.default_rng(seed)
    u = rng.random(size)
    return xmin * (1 - u) ** (-1 / (gamma - 1))


def sample_lognormal(mu=1.0, sigma=1.0, size=5000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    return rng.lognormal(mean=mu, sigma=sigma, size=size)


def popularity_summary(values, label):
    values = np.asarray(values, dtype=float)
    top10 = head_tail_share(values, head_fraction=0.10)
    return {
        'system': label,
        'n items': len(values),
        '<x>': values.mean(),
        'max x': values.max(),
        'Gini': gini_coefficient(values),
        'HHI': herfindahl_index(values),
        'top 10% share': top10['head_share'],
    }


def head_tail_table(values, label, fractions=(0.01, 0.05, 0.10, 0.20)):
    rows = []
    for frac in fractions:
        summary = head_tail_share(values, head_fraction=frac)
        rows.append({
            'system': label,
            'head fraction': frac,
            'head items': summary['head_items'],
            'head share': summary['head_share'],
            'tail share': summary['tail_share'],
        })
    return pd.DataFrame(rows)


---
## 1. A real popularity distribution: airport connectivity

In the slide deck, **popularity** is the outcome of unequal attention. In a transportation network, degree plays the same role: airports with many direct connections are more visible and more reachable.

We use the **OpenFlights world network** and treat airport degree as a simple popularity proxy.


In [ ]:
G_world = load_openflights_world()
degrees_world = np.array([degree for _, degree in G_world.degree()], dtype=int)

airports_world = pd.DataFrame({
    'IATA': list(G_world.nodes()),
    'airport': [G_world.nodes[node].get('name', node) for node in G_world.nodes()],
    'degree': degrees_world,
}).sort_values('degree', ascending=False).reset_index(drop=True)

display(airports_world.head(10))
display(pd.DataFrame([popularity_summary(degrees_world, 'OpenFlights world')]).round(3))


**Interpretation**
- A small number of airports act as global hubs.
- Most airports have far fewer direct connections.
- This already suggests a **broad** popularity distribution: the head is small, but the tail is long.


---
## 2. How broad is the tail?

A broad degree distribution is easier to see by comparison. We match the OpenFlights network to two reference models:

- **Erdos-Renyi**: narrow degree variability around the mean.
- **Barabasi-Albert**: growth plus preferential attachment, which creates hubs.

The question is not whether the real network is exactly one of these models. The question is which tail shape it resembles more closely.


In [ ]:
n_world = G_world.number_of_nodes()
m_world = G_world.number_of_edges()
mean_degree_world = 2 * m_world / n_world

G_er = nx.gnm_random_graph(n_world, m_world, seed=RANDOM_SEED)
G_ba = nx.barabasi_albert_graph(n_world, max(1, round(mean_degree_world / 2)), seed=RANDOM_SEED)

degrees_er = np.array([degree for _, degree in G_er.degree()], dtype=int)
degrees_ba = np.array([degree for _, degree in G_ba.degree()], dtype=int)

comparison_df = pd.DataFrame([
    popularity_summary(degrees_world, 'OpenFlights world'),
    popularity_summary(degrees_er, 'Matched ER'),
    popularity_summary(degrees_ba, 'Matched BA'),
])
display(comparison_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_ccdf(degrees_world, ax=axes[0], label='OpenFlights world', color=CATEGORY_PALETTE['blue'])
plot_ccdf(degrees_er, ax=axes[0], label='Matched ER', color=CATEGORY_PALETTE['orange'])
plot_ccdf(degrees_ba, ax=axes[0], label='Matched BA', color=CATEGORY_PALETTE['green'])
style_axis(axes[0], title='CCDF of airport popularity', xlabel='Degree k', ylabel='P(K >= k)', legend=True)

plot_rank_size(degrees_world, ax=axes[1], label='OpenFlights world', color=CATEGORY_PALETTE['blue'])
plot_rank_size(degrees_er, ax=axes[1], label='Matched ER', color=CATEGORY_PALETTE['orange'])
plot_rank_size(degrees_ba, ax=axes[1], label='Matched BA', color=CATEGORY_PALETTE['green'])
style_axis(axes[1], title='Rank-size view', xlabel='Rank', ylabel='Degree', legend=True)

plt.show()


**Interpretation**
- The **ER** tail collapses quickly: it has no serious hubs.
- The **BA** model is much broader and closer to the real network in its hub structure.
- The empirical network is therefore not just "random with noise". Its popularity is organized around strong heterogeneity.


---
## 3. Broad is not the same as power-law

The lecture distinction is important:

- **Heavy-tailed / broad** means that large values remain non-negligible.
- **Power-law** is a stronger functional claim: for large $x$, $p(x) \propto x^{-\gamma}$.
- **Long tail** asks a different question: how much total mass sits in the many low-popularity items?

The computational pitfall is clear: two broad distributions can look similar on a log-log plot over a limited range.


In [ ]:
pareto_sample = np.maximum(1, np.round(40 * sample_power_law(gamma=2.6, size=5000, seed=RANDOM_SEED))).astype(int)
lognormal_sample = np.maximum(1, np.round(40 * sample_lognormal(mu=0.8, sigma=1.1, size=5000, seed=RANDOM_SEED + 1))).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_ccdf(pareto_sample, ax=axes[0], label='Synthetic power law', color=CATEGORY_PALETTE['blue'])
plot_ccdf(lognormal_sample, ax=axes[0], label='Synthetic lognormal', color=CATEGORY_PALETTE['red'])
style_axis(axes[0], title='Two broad tails on a log-log CCDF', xlabel='Popularity x', ylabel='P(X >= x)', legend=True)

plot_rank_size(pareto_sample, ax=axes[1], label='Synthetic power law', color=CATEGORY_PALETTE['blue'])
plot_rank_size(lognormal_sample, ax=axes[1], label='Synthetic lognormal', color=CATEGORY_PALETTE['red'])
style_axis(axes[1], title='Rank-size comparison', xlabel='Rank', ylabel='Popularity', legend=True)

plt.show()

slope_df = pd.DataFrame({
    'k_min': [5, 10, 20, 40, 80],
    'Power-law naive CCDF slope': [naive_ccdf_slope(pareto_sample, kmin) for kmin in [5, 10, 20, 40, 80]],
    'Lognormal naive CCDF slope': [naive_ccdf_slope(lognormal_sample, kmin) for kmin in [5, 10, 20, 40, 80]],
})
display(slope_df.round(3))


**Interpretation**
- Both samples are broad.
- The power-law sample keeps a more stable tail slope.
- The lognormal sample can still look nearly linear over part of the log-log plot.

This is why good practice is to say **"consistent with a power law"** until a stronger statistical analysis is done.


---
## 4. Long tail: how much mass sits outside the head?

The **long-tail** question is not "is the tail broad?" It is:

> How much total popularity lives outside the small set of top items?

We answer it with two complementary views:
- a **rank-size** plot, which shows the head and the tail together,
- a **cumulative share** plot, which shows how much of the total is captured by the top ranks.


In [ ]:
world_head_tail = head_tail_table(degrees_world, 'OpenFlights world')
display(world_head_tail.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_rank_size(degrees_world, ax=axes[0], color=CATEGORY_PALETTE['blue'])
style_axis(axes[0], title='Rank-size curve of airport popularity', xlabel='Rank', ylabel='Degree')

plot_cumulative_share(degrees_world, ax=axes[1], color=CATEGORY_PALETTE['orange'])
for frac, color in [(0.01, CATEGORY_PALETTE['red']), (0.05, CATEGORY_PALETTE['purple']), (0.10, CATEGORY_PALETTE['green'])]:
    cutoff = int(np.ceil(frac * len(degrees_world)))
    axes[1].axvline(cutoff, color=color, linestyle='--', alpha=0.7)
style_axis(axes[1], title='How quickly the head captures total popularity', xlabel='Top-ranked airports kept', ylabel='Cumulative share')

plt.show()


**Interpretation**
- The head is very small, but it captures a large share of all connectivity.
- The tail is long: many airports remain peripheral.
- In infrastructure, that means **hub dependence**. In digital markets, the same geometry appears as **hits plus niches**.


---
## 5. Exercise

Compare the **USA** and **world** OpenFlights networks.

1. Which one is more concentrated by **Gini** and **top 10% share**?
2. Which one depends more strongly on a small set of hubs?
3. How would that change your robustness intuition from `NS07`?


In [ ]:
G_usa = load_openflights_usa()
degrees_usa = np.array([degree for _, degree in G_usa.degree()], dtype=int)

exercise_df = pd.DataFrame([
    popularity_summary(degrees_usa, 'OpenFlights USA'),
    popularity_summary(degrees_world, 'OpenFlights world'),
]).set_index('system')
display(exercise_df.round(3))

print('Top-10%-share comparison:')
print(exercise_df['top 10% share'].sort_values(ascending=False).round(3))


**Takeaway**
- `NS11` is not only about fitting straight lines on log-log axes.
- It is about understanding **why popularity is broad**, **when a power-law claim is too strong**, and **how the head/tail split changes the interpretation of a system**.
